# Taller 3 · Farmacia y medicamentos: análisis de dispensaciones

INSTRUCCIONES PARA EL ALUMNADO:  
1. Descarga el CSV del dataset "consumo-de-productos-farmaceuticos-por-receta"  
   desde analisis.datosabiertos.jcyl.es  
2. Guardalo como 'consumo_farmaceutico.csv' en la carpeta 'datos/' de este taller.  
3. Ejecuta las celdas en orden. Los CHECKPOINT verifican que vas bien.  


In [ ]:
# Celda 1 - Importar librerias
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Celda 2 - Cargar el dataset
try:
    df = pd.read_csv('datos/consumo_farmaceutico.csv', sep=';', encoding='utf-8')
except FileNotFoundError:
    raise FileNotFoundError(
        "No se encuentra 'datos/consumo_farmaceutico.csv'. "
        "Descargalo primero y colocalo en la carpeta 'datos/' de este taller."
    )

print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head()

In [ ]:
# Celda 3 - Exploracion inicial (Actividad 1)
print("Tipos de producto:", sorted(df['tipo_producto'].unique()))
print("Provincias:", sorted(df['provincia'].unique()))
print(f"Numero de tipos de producto: {df['tipo_producto'].nunique()}")

In [ ]:
# Celda 4 - Control de calidad (Actividad 2)
print("Valores nulos por columna:")
print(df.isnull().sum())

negativos = df[(df['numero_envases'] < 0) | (df['importe'] < 0)]
print(f"\nFilas con valores negativos (no deberian existir): {len(negativos)}")

df['mes'] = pd.to_datetime(df['mes'], format='%Y-%m', errors='coerce')
fechas_invalidas = df['mes'].isnull().sum()
print(f"Fechas que no se pudieron interpretar: {fechas_invalidas}")

In [ ]:
# CHECKPOINT 1
assert len(negativos) == 0, "Hay valores negativos en envases o importe, revisa el dataset original"
print("Checkpoint OK: no hay valores negativos")

In [ ]:
# Celda 5 - Envases totales por tipo de producto (Actividad 3)
por_tipo = df.groupby('tipo_producto')['numero_envases'].sum().sort_values(ascending=False)
print("Envases totales por tipo de producto:")
print(por_tipo)

plt.figure(figsize=(8, 5))
por_tipo.plot(kind='bar', color='seagreen')
plt.ylabel('Numero de envases')
plt.title('Consumo total por tipo de producto - Castilla y Leon')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('consumo_por_tipo.png', dpi=150)
plt.show()

In [ ]:
# Celda 6 - Desagregacion por provincia del producto mas consumido
producto_top = por_tipo.idxmax()
por_provincia = (
    df[df['tipo_producto'] == producto_top]
    .groupby('provincia')['numero_envases']
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(8, 5))
por_provincia.plot(kind='bar', color='seagreen')
plt.ylabel('Numero de envases')
plt.title(f'Consumo de {producto_top} por provincia')
plt.tight_layout()
plt.savefig('consumo_por_provincia.png', dpi=150)
plt.show()

In [ ]:
# Celda 7 - Patron estacional (Actividad 4)
producto_estacional = 'Medicamentos respiratorios'

if producto_estacional not in df['tipo_producto'].unique():
    print(f"Aviso: '{producto_estacional}' no esta en el dataset. Categorias disponibles: {df['tipo_producto'].unique()}")

df_estacional = df[df['tipo_producto'] == producto_estacional].copy()
df_estacional['mes_num'] = df_estacional['mes'].dt.month
estacional = df_estacional.groupby('mes_num')['numero_envases'].sum()

meses_nombre = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

plt.figure(figsize=(8, 5))
plt.plot(estacional.index, estacional.values, marker='o', color='darkorange', linewidth=2)
plt.xticks(range(1, 13), meses_nombre)
plt.xlabel('Mes')
plt.ylabel('Numero de envases (suma de todos los anios)')
plt.title(f'Patron estacional de consumo: {producto_estacional}')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('patron_estacional.png', dpi=150)
plt.show()

In [ ]:
# Celda 8 - Resumen automatico
mes_pico = meses_nombre[estacional.idxmax() - 1]
mes_valle = meses_nombre[estacional.idxmin() - 1]
print(f"Mes de mayor consumo de {producto_estacional}: {mes_pico}")
print(f"Mes de menor consumo de {producto_estacional}: {mes_valle}")

In [ ]:
# EJERCICIO - Completa tu mismo
# Repite la Celda 7 con otro tipo de producto (por ejemplo, antihistaminicos si esta disponible).
# ¿El pico de consumo se produce en un mes distinto? ¿Tiene sentido clinico esa diferencia?